# Hydrological signatures

In [ ]:
import pandas as pd
from tqdm import tqdm

from andeangc import config as cfg
from andeangc import hydro_signatures as hs

## Data

In [ ]:
AndeanGC_metadata = pd.read_csv(cfg.VERSION / 'AndeanGC_metadata.csv', index_col=0)
AndeanGC_data_qc  = pd.read_csv(cfg.VERSION / 'AndeanGC_data_qc.csv', index_col=0, parse_dates=True)
prcp = pd.read_parquet(cfg.VERSION / 'climate/historical/AndeanGC_prcp_ERA5.parquet')

# Signatures are computed over the climate attributes' reference period
# (config.yml:period_ref_climate), so that a signature and an attribute describe the same years.
period = slice(*cfg.period_ref_climate)
AndeanGC_data_qc = AndeanGC_data_qc.loc[period]
prcp = prcp.loc[period]

gauges = AndeanGC_metadata.index

In [ ]:
# m3/s -> mm/day
AndeanGC_data_qc = hs.to_mm_per_day(AndeanGC_data_qc, AndeanGC_metadata['basin_area'])

## Signatures

In [ ]:
records = {}

for gauge_id in tqdm(gauges):
    records[gauge_id] = hs.signatures(
        AndeanGC_data_qc[gauge_id],
        p                    = prcp[gauge_id],
        start_month          = cfg.water_year_start_month,
        min_coverage         = cfg.sig_min_coverage,
        min_years            = cfg.sig_min_years,
        bfi_alpha            = cfg.sig_bfi_alpha,
        bfi_passes           = cfg.sig_bfi_passes,
        high_flow_factor     = cfg.sig_high_flow_factor,
        low_flow_factor      = cfg.sig_low_flow_factor,
        fdc_percentiles      = cfg.sig_fdc_percentiles,
        recession_min_length = cfg.sig_recession_min_length,
        rolling_window       = cfg.sig_rolling_window)

AndeanGC_signatures = pd.DataFrame(records).T.rename_axis('gauge_id')
AndeanGC_signatures.describe().T.round(3)

In [ ]:
# Save
AndeanGC_signatures.to_csv(cfg.VERSION / 'AndeanGC_signatures.csv')